# Apply Sentiment Labels — Best Pipeline (Base → Pseudo → Human Fine-tune)

Trains the best-performing pipeline from ablation studies (Ablation 4: pseudo-label training only, no DAPT or FinnSentiment pre-training), then fine-tunes on all human-labeled data and applies labels to all unlabeled forum posts.

## Imports

In [9]:
import datetime
import gc
import os

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset as TorchDataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)
from datasets import Dataset
from sklearn.utils.class_weight import compute_class_weight
from google.colab import drive

## Mount Google Drive

In [10]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Config

In [11]:
BASE_MODEL        = 'TurkuNLP/bert-large-finnish-cased-v1'
NUM_LABELS        = 3
LABEL_NAMES       = ['negative', 'neutral', 'positive']
MAX_SEQ_LENGTH    = 512
BATCH_SIZE        = 256

timestamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

PSEUDO_MODEL_PATH = f'/tmp/{BASE_MODEL.split("/")[-1]}_pseudo_{timestamp}/'
FINAL_MODEL_PATH  = f'/content/drive/MyDrive/ColabThesisData/models/{BASE_MODEL.split("/")[-1]}_abl4_final_{timestamp}/'
LLM_LABELS_PATH   = '/content/drive/MyDrive/ColabThesisData/llm_labeled.parquet'
HUMAN_LABELS_PATH = '/content/drive/MyDrive/ColabThesisData/labeled.parquet'
ALL_POSTS_PATH    = '/content/drive/MyDrive/ColabThesisData/cleaned_forum_posts.parquet'
OUTPUT_PATH       = '/content/drive/MyDrive/ColabThesisData/all_labeled_finnishbert_new.parquet'

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device    : {DEVICE}')
print(f'Timestamp : {timestamp}')

Device    : cuda
Timestamp : 2026-04-13_02-20-06


In [12]:
!pip uninstall -y hf-xet
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"

## Load Tokenizer

In [13]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
print(f'Tokenizer loaded: {BASE_MODEL}')

Tokenizer loaded: TurkuNLP/bert-large-finnish-cased-v1


## Helpers

In [14]:
def make_hf_dataset(df):
    df = df.copy()
    df['text'] = 'yritys: ' + df['company_name'] + ' viesti: ' + df['message']
    hf = Dataset.from_pandas(df[['text', 'sentiment']].rename(columns={'sentiment': 'label'}).reset_index(drop=True))
    return hf.map(
        lambda batch: tokenizer(batch['text'], truncation=True, max_length=MAX_SEQ_LENGTH),
        batched=True,
    )


def ordinal_emd_loss(logits, labels, class_weights=None):
    """
    Ordinal Earth Mover's Distance (Wasserstein-1) loss for ordered classes.
    Penalizes mistakes in proportion to ordinal distance.
    Labels must be encoded as 0, 1, ..., K-1.
    """
    num_classes = logits.size(-1)

    if labels.dtype != torch.long:
        labels = labels.long()

    probs    = torch.softmax(logits, dim=-1)                          # (B, K)
    cum_pred = torch.cumsum(probs, dim=-1)[..., :-1]                  # (B, K-1)

    cum_true = torch.cumsum(
        torch.nn.functional.one_hot(labels, num_classes=num_classes).float(), dim=-1
    )[..., :-1]                                                        # (B, K-1)

    per_sample = torch.abs(cum_pred - cum_true).sum(dim=-1)           # (B,)

    if class_weights is not None:
        class_weights  = class_weights.to(logits.device)
        sample_weights = class_weights[labels]
        return (per_sample * sample_weights).sum() / sample_weights.sum()

    return per_sample.mean()

## Phase 1 — Pseudo-label Training

Loads `BASE_MODEL` directly (no DAPT, no FinnSentiment) and trains on LLM pseudo-labeled data.

In [15]:
llm_labels = pd.read_parquet(LLM_LABELS_PATH)
print(f'LLM pseudo-labels: {len(llm_labels):,}')
print(llm_labels['sentiment'].value_counts().sort_index().rename({0: 'negative', 1: 'neutral', 2: 'positive'}))

pseudo_ds = make_hf_dataset(llm_labels[['message', 'sentiment', 'company_name']])

LLM pseudo-labels: 10,000
sentiment
negative    3591
neutral     2782
positive    3627
Name: count, dtype: int64


Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [16]:
pseudo_model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL, num_labels=NUM_LABELS, device_map='auto', dtype=torch.bfloat16)

pseudo_args = TrainingArguments(
    output_dir='/tmp/pseudo_ckpts/',
    num_train_epochs=10,
    per_device_train_batch_size=16,
    learning_rate=3e-5, weight_decay=0.01, warmup_ratio=0.06,
    logging_steps=50, save_strategy='no',
    bf16=torch.cuda.is_available(), report_to='none', remove_unused_columns=True,
)
pseudo_trainer = Trainer(
    model=pseudo_model, args=pseudo_args,
    train_dataset=pseudo_ds,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
)

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


pytorch_model.bin:  19%|#8        | 268M/1.43G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
BertForSequenceClassification LOAD REPORT from: TurkuNLP/bert-large-finnish-cased-v1
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                   

In [17]:
pseudo_trainer.train()
pseudo_trainer.save_model(PSEUDO_MODEL_PATH)
tokenizer.save_pretrained(PSEUDO_MODEL_PATH)
print(f'Pseudo-label model saved to: {PSEUDO_MODEL_PATH}')

del pseudo_model, pseudo_trainer
gc.collect(); torch.cuda.empty_cache()

model.safetensors:   0%|          | 0.00/1.43G [00:00<?, ?B/s]

Step,Training Loss
50,1.138750
100,1.117864
150,1.094429
200,1.095962
250,1.071323
300,1.078845
350,1.007988
400,0.917419
450,0.839608
500,0.811347


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Pseudo-label model saved to: /tmp/bert-large-finnish-cased-v1_pseudo_2026-04-13_02-20-06/


## Phase 2 — Human-label Fine-tuning

Fine-tunes the pseudo-trained model on all available human-labeled data.

In [18]:
human_labels = pd.read_parquet(HUMAN_LABELS_PATH)
print(f'Human-labeled samples: {len(human_labels):,}')
print(human_labels['sentiment'].value_counts().sort_index().rename({0: 'negative', 1: 'neutral', 2: 'positive'}))

final_train_ds = make_hf_dataset(human_labels[['message', 'sentiment', 'company_name']])

Human-labeled samples: 608
sentiment
negative    150
neutral     253
positive    205
Name: count, dtype: int64


Map:   0%|          | 0/608 [00:00<?, ? examples/s]

In [19]:
final_model = AutoModelForSequenceClassification.from_pretrained(
    PSEUDO_MODEL_PATH, num_labels=NUM_LABELS, device_map='auto', dtype=torch.bfloat16)

final_cw = compute_class_weight('balanced', classes=np.array([0, 1, 2]), y=human_labels['sentiment'].values)
final_cw_tensor = torch.tensor(final_cw, dtype=torch.float).to(final_model.device)

class FinalWeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        loss = ordinal_emd_loss(outputs.logits, labels, class_weights=final_cw_tensor)
        return (loss, outputs) if return_outputs else loss

final_args = TrainingArguments(
    output_dir='/tmp/final_ckpts/',
    num_train_epochs=10,
    per_device_train_batch_size=16,
    learning_rate=3e-5, weight_decay=0.01, warmup_ratio=0.1,
    logging_steps=5, save_strategy='no',
    bf16=torch.cuda.is_available(), report_to='none', remove_unused_columns=True,
)
final_trainer = FinalWeightedTrainer(
    model=final_model, args=final_args,
    train_dataset=final_train_ds,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
)
final_trainer.train()

os.makedirs(FINAL_MODEL_PATH, exist_ok=True)
final_trainer.save_model(FINAL_MODEL_PATH)
tokenizer.save_pretrained(FINAL_MODEL_PATH)
print(f'Final model saved to: {FINAL_MODEL_PATH}')

del final_trainer
gc.collect(); torch.cuda.empty_cache()

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Step,Training Loss
5,0.510703
10,0.506301
15,0.587940
20,0.606637
25,0.582827
30,0.423064
35,0.446902
40,0.484030
45,0.476724
50,0.526198


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Final model saved to: /content/drive/MyDrive/ColabThesisData/models/bert-large-finnish-cased-v1_abl4_final_2026-04-13_02-20-06/


## Identify Unlabeled Posts

In [20]:
all_posts = pd.read_parquet(ALL_POSTS_PATH)

all_posts['id']    = all_posts['id'].astype(str)
llm_labels['id']   = llm_labels['id'].astype(str)
human_labels['id'] = human_labels['id'].astype(str)

already_labeled = set(llm_labels['id']) | set(human_labels['id'])
to_infer = all_posts[~all_posts['id'].isin(already_labeled)].reset_index(drop=True)

print(f'All forum posts  : {len(all_posts):,}')
print(f'Already labeled  : {len(already_labeled):,}')
print(f'Needs inference  : {len(to_infer):,}')

All forum posts  : 532,424
Already labeled  : 10,608
Needs inference  : 521,816


## Run Inference

In [21]:
class TextDataset(TorchDataset):
    def __init__(self, encodings):
        self.encodings = encodings
    def __len__(self):
        return self.encodings['input_ids'].shape[0]
    def __getitem__(self, idx):
        return {k: v[idx] for k, v in self.encodings.items()}


texts = ('yritys: ' + to_infer['company_name'] + ' viesti: ' + to_infer['message']).tolist()

encodings = tokenizer(
    texts,
    truncation=True,
    max_length=MAX_SEQ_LENGTH,
    padding=True,
    return_tensors='pt',
)
loader = DataLoader(TextDataset(encodings), batch_size=BATCH_SIZE)

final_model.eval()
all_preds = []
total = len(loader)

with torch.no_grad():
    for i, batch in enumerate(loader, 1):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        preds = torch.argmax(final_model(**batch).logits, dim=-1).cpu().numpy()
        all_preds.extend(preds)
        if i % 50 == 0 or i == total:
            print(f'  {i}/{total} batches done')

print(f'Inference complete: {len(all_preds):,} predictions')

  50/2039 batches done
  100/2039 batches done
  150/2039 batches done
  200/2039 batches done
  250/2039 batches done
  300/2039 batches done
  350/2039 batches done
  400/2039 batches done
  450/2039 batches done
  500/2039 batches done
  550/2039 batches done
  600/2039 batches done
  650/2039 batches done
  700/2039 batches done
  750/2039 batches done
  800/2039 batches done
  850/2039 batches done
  900/2039 batches done
  950/2039 batches done
  1000/2039 batches done
  1050/2039 batches done
  1100/2039 batches done
  1150/2039 batches done
  1200/2039 batches done
  1250/2039 batches done
  1300/2039 batches done
  1350/2039 batches done
  1400/2039 batches done
  1450/2039 batches done
  1500/2039 batches done
  1550/2039 batches done
  1600/2039 batches done
  1650/2039 batches done
  1700/2039 batches done
  1750/2039 batches done
  1800/2039 batches done
  1850/2039 batches done
  1900/2039 batches done
  1950/2039 batches done
  2000/2039 batches done
  2039/2039 batches 

## Combine & Save

In [22]:
inferred_df = to_infer.copy()
inferred_df['sentiment']    = all_preds
inferred_df['label_source'] = 'model'

llm_out             = llm_labels.copy()
llm_out['label_source'] = 'llm'

human_out               = human_labels.copy()
human_out['label_source'] = 'human'

keep_cols = ['id', 'message', 'company_name', 'sentiment', 'label_source']

combined = pd.concat(
    [inferred_df[keep_cols], llm_out[keep_cols], human_out[keep_cols]],
    ignore_index=True,
)

print(f'Total rows : {len(combined):,}')
print(combined['label_source'].value_counts())
print()
print(combined['sentiment'].value_counts().sort_index().rename({0: 'negative', 1: 'neutral', 2: 'positive'}))

combined.to_parquet(OUTPUT_PATH, index=False)
print(f'\nSaved to: {OUTPUT_PATH}')

del final_model
gc.collect()
torch.cuda.empty_cache()

Total rows : 532,424
label_source
model    521816
llm       10000
human       608
Name: count, dtype: int64

sentiment
negative    142001
neutral     202611
positive    187812
Name: count, dtype: int64

Saved to: /content/drive/MyDrive/ColabThesisData/all_labeled_finnishbert_new.parquet
